In [1]:
# --- headless execution harness (added by run_experiments.py) ---
import os as _h_os
import matplotlib as _h_mpl
_h_mpl.use('Agg')
import matplotlib.pyplot as _h_plt
_OUT = 'C:\\Users\\ASUS\\group_sgu26_deep_learning\\BTL03\\output\\2. CNN + MLP\\1_MNIST_CNN_pytorch'
_h_os.makedirs(_OUT, exist_ok=True)
_h_n = [0]
_h_show = _h_plt.show
def _saving_show(*args, **kwargs):
    _figs = [_h_plt.figure(n) for n in _h_plt.get_fignums()]
    for _f in _figs:
        _h_n[0] += 1
        _f.savefig(_h_os.path.join(_OUT, 'fig_%02d.png' % _h_n[0]), dpi=110, bbox_inches='tight')
    _h_show(*args, **kwargs)
    for _f in _figs:
        _h_plt.close(_f)
_h_plt.show = _saving_show
print('[harness] figures -> ' + _OUT)


[harness] figures -> C:\Users\ASUS\group_sgu26_deep_learning\BTL03\output\2. CNN + MLP\1_MNIST_CNN_pytorch


In [2]:
# Demo (PyTorch)

# Overview
- MNIST dataset
  + Goal: recognizing handwritten digits
  + Input: a 28x28 image (PyTorch format: `(1, 28, 28)` - NCHW)
  + Output: class label (0, 1,..., 9)
- Defining a CNN+MLP for MNIST in PyTorch
- Training, evaluating, and visualizing PCA features (`raw pixels`, `fc2`, `pool3 + GAP`)

# Import libraries

In [3]:
import time
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from torchvision import datasets

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

Using device: cuda


In [4]:
num_classes = 10   # 10 classes: 0, 1,..., 9

# (1) Data

In [5]:
# Load MNIST via torchvision
train_raw = datasets.MNIST(root='./data', train=True, download=True)
test_raw = datasets.MNIST(root='./data', train=False, download=True)

# PyTorch Conv2d yêu cầu định dạng NCHW: (N, C, H, W) = (N, 1, 28, 28)
x_train = train_raw.data.numpy().reshape(60000, 1, 28, 28).astype('float32') / 255.0
y_train = train_raw.targets.numpy().astype('int64')

x_test = test_raw.data.numpy().reshape(10000, 1, 28, 28).astype('float32') / 255.0
y_test = test_raw.targets.numpy().astype('int64')

print(x_train.shape[0], 'train samples | shape:', x_train.shape)
print(x_test.shape[0], 'test samples  | shape:', x_test.shape)

60000 train samples | shape: (60000, 1, 28, 28)
10000 test samples  | shape: (10000, 1, 28, 28)


In [6]:
# Minh họa one-hot trong PyTorch (tương đương keras.utils.to_categorical(2, num_classes=3))
# Lưu ý: CrossEntropyLoss của PyTorch nhận trực tiếp nhãn số nguyên (0..9)
F.one_hot(torch.tensor(2), num_classes=3).float().numpy()

array([0., 0., 1.], dtype=float32)

# (2) Declare model

In [7]:
class MNIST_CNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        # Block 1: (1, 28, 28) -> (64, 14, 14)
        self.conv1 = nn.Conv2d(1, 64, kernel_size=3, padding=1)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.conv_dropout1 = nn.Dropout(p=0.2)

        # Block 2: (64, 14, 14) -> (128, 7, 7)
        self.conv2 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.conv_dropout2 = nn.Dropout(p=0.2)

        # Block 3: (128, 7, 7) -> (256, 3, 3)
        self.conv3 = nn.Conv2d(128, 256, kernel_size=3, padding=1)
        self.pool3 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.conv_dropout3 = nn.Dropout(p=0.2)

        self.flatten = nn.Flatten()
        # 256 * 3 * 3 = 2304
        self.fc1 = nn.Linear(256 * 3 * 3, 512)
        self.fc_dropout1 = nn.Dropout(p=0.2)
        self.fc2 = nn.Linear(512, 512)
        self.fc_dropout2 = nn.Dropout(p=0.2)
        self.fc3_10ways = nn.Linear(512, num_classes)

    def forward(self, x, return_layer=None):
        x = self.conv_dropout1(self.pool1(F.relu(self.conv1(x))))
        x = self.conv_dropout2(self.pool2(F.relu(self.conv2(x))))
        pool3_out = self.pool3(F.relu(self.conv3(x)))
        if return_layer == 'pool3_gap':
            # GlobalAveragePooling2D tương đương AdaptiveAvgPool2d(1) / mean(dim=(2,3))
            return pool3_out.mean(dim=(2, 3))
        x = self.conv_dropout3(pool3_out)
        x = self.flatten(x)
        x = self.fc_dropout1(F.relu(self.fc1(x)))
        fc2_out = F.relu(self.fc2(x))
        if return_layer == 'fc2':
            return fc2_out
        x = self.fc_dropout2(fc2_out)
        return self.fc3_10ways(x)

model = MNIST_CNN(num_classes=num_classes).to(device)
print(model)
total_params = sum(p.numel() for p in model.parameters())
print(f'Total params: {total_params:,}')  # Khớp chính xác 1,817,610 params của bản Keras

MNIST_CNN(
  (conv1): Conv2d(1, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool1): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv_dropout1): Dropout(p=0.2, inplace=False)
  (conv2): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv_dropout2): Dropout(p=0.2, inplace=False)
  (conv3): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv_dropout3): Dropout(p=0.2, inplace=False)
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (fc1): Linear(in_features=2304, out_features=512, bias=True)
  (fc_dropout1): Dropout(p=0.2, inplace=False)
  (fc2): Linear(in_features=512, out_features=512, bias=True)
  (fc_dropout2): Dropout(p=0.2, inplace=False)
  (fc3_10ways): Linear(in_features=512, out_features=10, bias=True)
)
Total params: 1,

In [8]:
try:
    from torchinfo import summary
    print(summary(model, input_size=(1, 1, 28, 28), device=device))
except ImportError:
    print('Cài đặt torchinfo nếu muốn xem bảng summary chi tiết: pip install torchinfo')

Layer (type:depth-idx)                   Output Shape              Param #
MNIST_CNN                                [1, 10]                   --
├─Conv2d: 1-1                            [1, 64, 28, 28]           640
├─MaxPool2d: 1-2                         [1, 64, 14, 14]           --
├─Dropout: 1-3                           [1, 64, 14, 14]           --
├─Conv2d: 1-4                            [1, 128, 14, 14]          73,856
├─MaxPool2d: 1-5                         [1, 128, 7, 7]            --
├─Dropout: 1-6                           [1, 128, 7, 7]            --
├─Conv2d: 1-7                            [1, 256, 7, 7]            295,168
├─MaxPool2d: 1-8                         [1, 256, 3, 3]            --
├─Dropout: 1-9                           [1, 256, 3, 3]            --
├─Flatten: 1-10                          [1, 2304]                 --
├─Linear: 1-11                           [1, 512]                  1,180,160
├─Dropout: 1-12                          [1, 512]                  -

# (3) Train defined model

In [9]:
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for bx, by in loader:
            bx, by = bx.to(device), by.to(device)
            logits = model(bx)
            loss = criterion(logits, by)
            total_loss += loss.item() * bx.size(0)
            correct += (logits.argmax(dim=1) == by).sum().item()
            total += bx.size(0)
    return total_loss / total, correct / total


lr = 0.01
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=lr)

batch_size = 128
epochs = 30

train_loader = DataLoader(
    TensorDataset(torch.from_numpy(x_train), torch.from_numpy(y_train)),
    batch_size=batch_size, shuffle=True
)
test_loader = DataLoader(
    TensorDataset(torch.from_numpy(x_test), torch.from_numpy(y_test)),
    batch_size=batch_size, shuffle=False
)

history = {'loss': [], 'accuracy': [], 'val_loss': [], 'val_accuracy': []}

starting_time = time.time()
for epoch in range(1, epochs + 1):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for bx, by in train_loader:
        bx, by = bx.to(device), by.to(device)
        optimizer.zero_grad()
        logits = model(bx)
        loss = criterion(logits, by)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * bx.size(0)
        correct += (logits.argmax(dim=1) == by).sum().item()
        total += bx.size(0)

    train_loss = running_loss / total
    train_acc = correct / total
    val_loss, val_acc = evaluate(model, test_loader, criterion, device)

    history['loss'].append(train_loss)
    history['accuracy'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_accuracy'].append(val_acc)

    print(f'Epoch {epoch:2d}/{epochs} - loss: {train_loss:.4f} - accuracy: {train_acc:.4f} - val_loss: {val_loss:.4f} - val_accuracy: {val_acc:.4f}')

print('> training time is %.4f minutes' % ((time.time() - starting_time) / 60))

Epoch  1/30 - loss: 2.2970 - accuracy: 0.1593 - val_loss: 2.2884 - val_accuracy: 0.3036


Epoch  2/30 - loss: 2.2530 - accuracy: 0.2780 - val_loss: 2.1078 - val_accuracy: 0.4925


Epoch  3/30 - loss: 1.2859 - accuracy: 0.5957 - val_loss: 0.6798 - val_accuracy: 0.7826


Epoch  4/30 - loss: 0.6465 - accuracy: 0.7903 - val_loss: 0.3415 - val_accuracy: 0.8980


Epoch  5/30 - loss: 0.4069 - accuracy: 0.8717 - val_loss: 0.2181 - val_accuracy: 0.9357


Epoch  6/30 - loss: 0.2887 - accuracy: 0.9096 - val_loss: 0.1755 - val_accuracy: 0.9452


... (20 outputs left out)

Epoch 27/30 - loss: 0.0695 - accuracy: 0.9788 - val_loss: 0.0377 - val_accuracy: 0.9867


Epoch 28/30 - loss: 0.0673 - accuracy: 0.9791 - val_loss: 0.0363 - val_accuracy: 0.9877


Epoch 29/30 - loss: 0.0663 - accuracy: 0.9791 - val_loss: 0.0372 - val_accuracy: 0.9872


Epoch 30/30 - loss: 0.0645 - accuracy: 0.9800 - val_loss: 0.0345 - val_accuracy: 0.9876
> training time is 4.3103 minutes


# (4) Evaluate trained model & PCA Visualization

In [10]:
test_loss, test_acc = evaluate(model, test_loader, criterion, device)
print('Test loss:', test_loss)
print('Test accuracy:', test_acc)

Test loss: 0.03451622300334275
Test accuracy: 0.9876


In [11]:
# PCA trên ảnh gốc (60000, 784)
data_org = x_train.reshape(x_train.shape[0], -1)
data_lbl = y_train
print(data_org.shape, data_lbl.shape)

pca = PCA(n_components=2)
org_pca = pca.fit_transform(data_org)
print(org_pca.shape)

plt.figure(figsize=(8, 8))
sns.scatterplot(x=org_pca[:, 0], y=org_pca[:, 1], hue=data_lbl, palette='viridis', s=10)
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title('Visualization of MNIST raw pixels using PCA', fontsize=15)
plt.show()

(60000, 784) (60000,)


(60000, 2)


C:\Users\ASUS\AppData\Local\Temp\ipykernel_6116\2374272517.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  _h_show(*args, **kwargs)


In [12]:
# select a test image randomly
random_test_index = np.random.choice(x_test.shape[0], size=1)[0]
test_img = x_test[random_test_index]   # shape: (1, 28, 28)
test_label = int(y_test[random_test_index])

model.eval()
with torch.no_grad():
    inp = torch.from_numpy(np.expand_dims(test_img, axis=0)).to(device)
    pred_label = int(model(inp).argmax(dim=1).item())

plt.imshow(test_img[0], cmap='gray')
plt.title('true label = %d, predicted label = %d' % (test_label, pred_label))
plt.show()

C:\Users\ASUS\AppData\Local\Temp\ipykernel_6116\2374272517.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  _h_show(*args, **kwargs)


In [13]:
# Hàm trích xuất đặc trưng theo batch (an toàn VRAM trên GPU 4GB)
def extract_features(model, x_array, layer_name, batch_size=256, device=device):
    model.eval()
    loader = DataLoader(TensorDataset(torch.from_numpy(x_array)), batch_size=batch_size, shuffle=False)
    feats = []
    with torch.no_grad():
        for (bx,) in loader:
            bx = bx.to(device)
            feats.append(model(bx, return_layer=layer_name).cpu().numpy())
    return np.concatenate(feats, axis=0)

In [14]:
# PCA trên đặc trưng tầng fc2 (60000, 512)
y_feat = extract_features(model, x_train, layer_name='fc2')
print('fc2 features shape:', y_feat.shape)

pca_fc2 = PCA(n_components=2)
fc2_pca = pca_fc2.fit_transform(y_feat)
print('fc2 PCA shape:', fc2_pca.shape)

plt.figure(figsize=(8, 8))
sns.scatterplot(x=fc2_pca[:, 0], y=fc2_pca[:, 1], hue=data_lbl, palette='viridis', s=10)
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title('Visualization of MNIST fc2 features using PCA', fontsize=15)
plt.show()

fc2 features shape: (60000, 512)


fc2 PCA shape: (60000, 2)


C:\Users\ASUS\AppData\Local\Temp\ipykernel_6116\2374272517.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  _h_show(*args, **kwargs)


In [15]:
# PCA trên đặc trưng pool3 + GlobalAveragePooling2D (60000, 256)
y2_feat = extract_features(model, x_train, layer_name='pool3_gap')
print('pool3 + GAP features shape:', y2_feat.shape)

pca_y2 = PCA(n_components=2)
y2_pca = pca_y2.fit_transform(y2_feat)
print('pool3 + GAP PCA shape:', y2_pca.shape)

plt.figure(figsize=(8, 8))
sns.scatterplot(x=y2_pca[:, 0], y=y2_pca[:, 1], hue=data_lbl, palette='viridis', s=10)
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title('Visualization of MNIST pool3+GAP features using PCA', fontsize=15)
plt.show()

pool3 + GAP features shape: (60000, 256)


pool3 + GAP PCA shape: (60000, 2)


C:\Users\ASUS\AppData\Local\Temp\ipykernel_6116\2374272517.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  _h_show(*args, **kwargs)
